
# 練習問題: omp simd でループをSIMD化する

## 目標

`#pragma omp simd` (Fortran では `!$omp simd`) を1つ挿入するだけで, コンパイラがループをSIMD命令 (`pd` 付きの packed double 命令) に変換することを体験する.

## 課題

`omp_simd.cpp` (または `omp_simd.f90`) は, saxpy/axpy と呼ばれる典型的な演算 `y[i] = a * x[i] + y[i]` を行う関数である.
配列 `x`, `y` が重なっている (エイリアスしている) 可能性をコンパイラが排除できないと, 安全のため自動ベクトル化を諦めてしまうことがある.

コメント `TODO` の指示に従って **OpenMP の指示行を1つ追加** し, ループがSIMD化されるようにせよ.

- C++: ループの直前に `#pragma omp simd` を1行加える.
- Fortran: doループの直前に `!$omp simd` を1行加える.

それ以外のコードを変更する必要はない.

## コンパイルと実行

`#pragma omp simd` を解釈させるには `-mp=multicore` が必要である (SIMD命令を使うだけなので1スレッドで動作する).

```
# C++
nvc++ -fast -mp=multicore -Mkeepasm -Minfo -Mneginfo -c omp_simd.cpp

# Fortran
nvfortran -fast -mp=multicore -Mkeepasm -Minfo -Mneginfo -c omp_simd.f90
```

- `-Mkeepasm` : 生成されたアセンブリ言語のファイル (`omp_simd.s`) を残す
- `-Minfo` / `-Mneginfo` : SIMD化に成功した・失敗したことを報告してくれる

生成されたアセンブリを確認する.

```
cat omp_simd.s
```

## 期待される結果

`#pragma omp simd` (`!$omp simd`) を追加すると, `omp_simd.s` の中に積和演算のSIMD命令 `vfmadd...pd` (または `vmulpd` + `vaddpd`) が現れる.
命令末尾の `pd` は _packed double precision_ の略で, _p_ (packed) がSIMD命令であることの証しである.
また `-Minfo` の出力に "Generated vector ..." のようなベクトル化成功のメッセージが現れることを確認せよ.

指示行を追加する前後でアセンブリ (`omp_simd.s`) や `-Minfo` の出力がどう変わるかを比べてみよ.



# 1. このノートについて
- この問題は「生成されたアセンブリ(機械語)を見て, SIMD命令に変換されたかを確認する」ことが目的。
- 関数だけのソース(`main` 無し)を `-c` でオブジェクトコンパイルし, `-Mkeepasm` で残る `.s` を読む。
- 実行はしない(`-c` なので実行ファイルは作らない)。

# 2. ツールの読み込み
- AIチュータ及びジョブ投入ツールの読み込み (カーネル起動後に一度実行すればよい)
  - `heytutor` : `%%hey` でAIチュータに質問できるようになる (使い方は末尾を参照)


In [ ]:
import heytutor
import wisteria_submit

# 3. C++ ベースコード

In [ ]:
%%writefile_ omp_simd.cpp
/* y[i] = a * x[i] + y[i] (saxpy/axpy) を n 要素について行う */
void saxpy(long n, double a, double * x, double * y) {
  // TODO: 下の for ループの直前に #pragma omp simd を1行追加し, このループをSIMD化せよ.
  for (long i = 0; i < n; i++) {
    y[i] = a * x[i] + y[i];
  }
}

## 3-1. コンパイル (アセンブリを残す)
- `-c` : 実行ファイルを作らずオブジェクトファイルまで (`main` が無くてよい)
- `-Mkeepasm` : 生成アセンブリ `omp_simd.s` を残す
- `-Minfo` / `-Mneginfo` : ベクトル化に成功/失敗したことを報告する

In [ ]:
%%bash_
PATH=/work/opt/local/x86_64/cores/nvidia/23.3/Linux_x86_64/23.3/compilers/bin:/opt/FJSVxtclanga/tcsds-1.2.41/bin:$PATH
nvc++ -fast -mp=multicore -Mkeepasm -Minfo -Mneginfo -c omp_simd.cpp

## 3-2. 生成されたアセンブリを見る
- `pd` の付いた packed 命令 (`vmulpd`, `vaddpd`, `vfmadd...pd`) や, `%ymm`/`%zmm` レジスタが出ていればSIMD化されている。

In [ ]:
%%bash_
cat omp_simd.s

# 4. Fortran ベースコード

In [ ]:
%%writefile_ omp_simd.f90
! y(i) = a * x(i) + y(i) (saxpy/axpy) を n 要素について行う
subroutine saxpy(n, a, x, y)
  implicit none
  integer(8), intent(in) :: n
  real(8), intent(in) :: a
  real(8), intent(in) :: x(n)
  real(8), intent(inout) :: y(n)
  integer(8) :: i
  ! TODO: 下の do ループの直前に !$omp simd を1行追加し, このループをSIMD化せよ.
  do i = 1, n
     y(i) = a * x(i) + y(i)
  end do
end subroutine saxpy

## 4-1. コンパイル (アセンブリを残す)

In [ ]:
%%bash_
PATH=/work/opt/local/x86_64/cores/nvidia/23.3/Linux_x86_64/23.3/compilers/bin:/opt/FJSVxtclanga/tcsds-1.2.41/bin:$PATH
nvfortran -fast -mp=multicore -Mkeepasm -Minfo -Mneginfo -c omp_simd.f90

## 4-2. 生成されたアセンブリを見る

In [ ]:
%%bash_
cat omp_simd.s


# 5. AIチュータへの質問の仕方 (参考)
- 先頭で `import heytutor` 済みなら, セルに `%%hey` と書いて質問できる。
- ChatGPTなどと同様に自由に質問してよい。ただし「答えをそのまま教えて」などは自制すること。
- セル内で使える変数 (自動で展開される):
  - `{file:FILENAME}` : _FILENAME_ の中身 (例: `{file:problem.md}`, `{file:omp_simd.cpp}`, `{file:omp_simd.s}`)
  - `{bash[-1]}` : 最後に実行した `%%bash_` セルの入力・出力, `{bash[-2]}` = その前, ...
- 以下は質問例 (必要に応じてコピーして使う。Fortranなら `.cpp` を `.f90` に書き換える)。

## 5-1. 一般的な質問

In [ ]:
%%hey

SIMD命令(packed double, vfmadd...pd など)って何?

## 5-2. この問題に関するヒント

In [ ]:
%%hey

この問題に関するヒントを教えて

問題:
{file:problem.md}

## 5-3. 困ったときのヘルプ
- コンパイル時のエラー直後に実行するとエラーに関するヘルプが得られる。

In [ ]:
%%hey

以下のエラーが出た。何が間違い?

プログラム:
{file:omp_simd.cpp}

コマンドと実行結果:
{bash[-1]}

## 5-4. アセンブリについて質問

In [ ]:
%%hey

生成されたアセンブリを説明して。SIMD化されている?

ソース:
{file:omp_simd.cpp}

アセンブリ:
{file:omp_simd.s}